#  Lab 3: Build a RAG PDF Chatbot

Welcome to Lab 3! This is the culmination of everything we've learned in this series. Now you'll build a complete **Retrieval Augmented Generation** application that can:

-  Upload and process PDF documents
-  Convert document chunks into embeddings
-  Store vectors in Qdrant database
-  Chat with your documents using Gemini
-  Present it all in a beautiful Streamlit UI

**What you'll learn:**
- How RAG works end-to-end
- PDF text extraction and chunking
- Combining vector search with LLM generation
- Building interactive AI applications with Streamlit

**Prerequisites:**
- Completed Lab 1 (LLM basics) and Lab 2 (Vector databases)
- A configured Google backend from the README
- Local Qdrant; no account is required

---


##  Understanding RAG (Retrieval Augmented Generation)

### What is RAG?

**RAG** is a technique that enhances LLM responses by providing relevant context from your own documents. Instead of relying solely on the LLM's training data, RAG retrieves specific information to answer questions accurately.

### How RAG Works (The Pipeline):

```

│  1. INGESTION (One-time setup)                                  │
│     PDF → Extract Text → Split into Chunks → Create Embeddings  │
│           → Store in Vector Database                            │
 -----------------------------------------------------------------
                              ↓

│  2. RETRIEVAL (When user asks a question)                       │
│     User Question → Create Embedding → Search Vector DB         │
│           → Get Top-K Similar Chunks                            │
 -----------------------------------------------------------------
                              ↓

│  3. GENERATION (Create the answer)                              │
│     Retrieved Chunks + User Question → Send to LLM → Answer     │
 -----------------------------------------------------------------
```

### Why RAG?

|        Without RAG              |               With RAG                        |
|---------------------------------|-----------------------------------------------|
| LLM only knows training data    | LLM has access to **your specific documents** |
| May hallucinate facts           | Can use document evidence; still check support and citations       |
| Can't answer about private data | Can use extractable text from supported documents    |
| Generic responses               | **Specific, accurate** responses              |

---


## Step 1: Environment setup

Use the pinned requirements from the README. This lab uses Google Gen AI for generation, the local embedding model from Lab 2, Qdrant, and pypdf. A small fictional handbook is included, so no PDF upload is needed to follow the notebook.


In [ ]:
# Install requirements.txt from the README before opening this notebook.
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "lab_support.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the cloned AI_Trainings repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT.name)


##  Step 2: Import Libraries

Let's import all the libraries we need. Each serves a specific purpose in our RAG pipeline.


In [ ]:
from lab_support import RagSession, get_client, DEFAULT_MODEL, split_text


## Step 3: Configure services

Use the same Google backend as Lab 1. Embeddings run locally. Retrieved PDF passages and questions are sent to Google during generation; use the supplied fictional sample for practice.


In [ ]:
client = get_client()
MODEL_NAME = DEFAULT_MODEL


### Storage

Qdrant is local and ephemeral by default. Hosted Qdrant is optional through environment variables. Each session creates its own collection and never recreates an existing collection.


In [ ]:
rag = RagSession(client=client, model=MODEL_NAME)
print("Session collection:", rag.collection)


### Chunking and retrieval settings

Chunk size is measured in characters. Overlap keeps some neighboring context. Top-k is the number of passages retrieved; a higher value can also introduce irrelevant material.


In [ ]:
CHUNK_SIZE = 500
OVERLAP = 80
TOP_K = 3


## Step 4: Inspect the RAG pipeline

The shared `lab_support.py` implementation is used by both the notebook and Streamlit app. This avoids copying credentials or maintaining two different retrieval pipelines. Read each method below before running it.


In [ ]:
import inspect
print(inspect.getsource(split_text))
print(inspect.getsource(RagSession.__init__))


### PDF ingestion

Text is extracted page by page. Each chunk carries the filename and page number, is embedded locally, and is inserted into the session collection. Scanned PDFs without text produce an explicit error; OCR is outside this lesson.


In [ ]:
print(inspect.getsource(RagSession.ingest))


### Retrieve, then generate

The model receives numbered excerpts and an instruction to abstain when they do not answer the question. These instructions do not guarantee factual correctness. Inspect the actual retrieved sources and verify the answer.


In [ ]:
print(inspect.getsource(RagSession.retrieve))
print(inspect.getsource(RagSession.answer))


---

##  Step 5: Test the RAG Pipeline (Notebook Version)

Before running the full Streamlit app, let's test each component in the notebook.

### 5.1 Test: Ingest a PDF

First, let's ingest a sample PDF. You can use any PDF file you have available.


In [ ]:
PDF_PATH = ROOT / "data" / "sample_handbook.pdf"
count = rag.ingest([PDF_PATH], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
assert count > 0
print("Indexed passages:", count)
hits = rag.retrieve("How long is the return window?", k=TOP_K)
for i, hit in enumerate(hits, 1):
    print(i, hit.payload["source"], "page", hit.payload["page"], hit.payload["text"])


### 5.2 Test: Ask a Question

After ingesting a PDF, test the RAG pipeline by asking a question about the document.


In [ ]:
TEST_QUESTION = "How long is the return window?"
result = rag.answer(TEST_QUESTION, k=TOP_K)
print(result["answer"])
print("Sources:", [(s["source"], s["page"]) for s in result["sources"]])
assert result["sources"]
# Manually verify: the fictional handbook says 30 days with a receipt.
# Also ask about a policy missing from the handbook and check for abstention.


## Step 6: Launch the Streamlit UI

The app is checked into the repository and reads the same environment variables. Run the printed command in a separate terminal from the repository root. It will keep running until you stop it with Ctrl+C. In the UI, choose the sample PDF, click **Index documents**, then ask about the return window.


In [ ]:
assert (ROOT / "rag_chatbot_app.py").is_file()
print("The app shares RagSession with this notebook; no code or credentials are generated.")


In [ ]:
print("python -m streamlit run rag_chatbot_app.py")


### Expected behavior

The chat stays disabled until a text PDF has been indexed. Answers show retrieved passages with page references for checking. Changing the selected documents clears the old index and chat history. Each question is answered independently; this lesson does not implement conversational query rewriting.


## Exercises

### 1. Compare chunk sizes

Try 250, 500, and 1000 characters. Keep the question fixed, inspect retrieved passages, and explain whether context was split or diluted. This exercise only retrieves; it makes no model calls.


In [ ]:
CHUNK_SIZES_TO_TEST = [250, 500, 1000]
for size in CHUNK_SIZES_TO_TEST:
    trial = RagSession(client=client)
    try:
        trial.ingest([PDF_PATH], chunk_size=size, overlap=80)
        passages = trial.retrieve(TEST_QUESTION, k=3)
        print(size, [p.payload["text"] for p in passages])
    finally:
        trial.close()


### 2. Compare top-k

Retrieve 1, 3, and 5 passages for the same question. Which ones contain useful evidence? How would extra passages affect cost and relevance? A small document may contain fewer than k passages.


In [ ]:
K_VALUES_TO_TEST = [1, 3, 5]
for k in K_VALUES_TO_TEST:
    print(k, [h.payload for h in rag.retrieve(TEST_QUESTION, k=k)])


### 3. Change style without losing grounding

Use the `system_instruction` argument to request a concise answer. Then ask a question the PDF cannot answer. Discuss why an instruction alone cannot prove factuality. Generation calls may incur charges.


In [ ]:
print(rag.answer(TEST_QUESTION, system_instruction="Use at most two sentences.")["answer"])


## What you built

A page-aware PDF retrieval pipeline with local embeddings, Qdrant, and Gemini generation. Limitations include character-based chunking, no OCR, no reranker, and no guarantee of correct citations. Try a missing-answer question and a misleading retrieved passage before extending it.

When finished, run `rag.close()` to remove this session collection. Stop Streamlit with Ctrl+C. Hosted Qdrant users should verify their practice collections are removed.
